# Day 13 ML/RL Prep

Broader PyTorch mix, still RL-centered:
- 1 code-reading / shape question
- 2 tiny implementation prompts
- 1 debugging question
- 1 short explanation question

Instructions:
- Keep answers short and explicit.
- State key shapes at important steps.
- Use RL as the setting, but focus on PyTorch fluency.


## Question 1: Read the transpose / permute

```python
x = torch.randn(2, 3, 4)
y = x.transpose(1, 2)
z = x.permute(2, 0, 1)
```

What are the shapes of `y` and `z`?
What does each operation do?
How is `transpose` different from `permute` here?


In [3]:
# Question 1: Answer and reasoning
import torch

x = torch.randn(2, 3, 4)
y = x.transpose(1, 2)
z = x.permute(2, 0, 1)
x.shape, y.shape, z.shape

# y shape: (2, 4, 3) as the dimentions at the second and third position are swapped.
# z shape: (4, 2, 3) as the permute reorders the dimentions based uppon its params. 
    # We asked x.index:3 to go into z index 1, etc.
# Transpose just swaps two dimentions, whereas permute is a full re-ordering.

(torch.Size([2, 3, 4]), torch.Size([2, 4, 3]), torch.Size([4, 2, 3]))

## Question 2: Tiny implementation

You have:
- `values`: shape `(B, T)`

You want to normalize values within each trajectory to have mean 0.

Do not over-polish the code. I want:
- what mean you would compute
- whether you would keep dimensions or not
- what the output shape would be
- what one row means after normalization


In [ ]:
# Question 2: Answer and reasoning

# You compute the mean across the time dimension (dimension 1).
mean = values.mean(dim=1, keepdim=True)

# Yes, you should keep dimensions (keepdim=True).
# If you don't, the mean shape becomes (B,). To subtract a (B,) from a (B, T), 
# you'd have to manually reshape it. By keeping the dims, the shape is (B, 1), 
# which broadcasts perfectly across all T time steps automatically.

# The output shape remains (B, T).
# The subtraction values - mean doesn't change the number of elements; it just shifts the values in each row.

# One row represents the relative rewards/values for that specific trajectory.

## Question 3: Tiny implementation

You have a list of trajectory reward tensors, each with shape `(T,)`, and you want one batch tensor with shape `(B, T)`.

Should you use `torch.stack` or `torch.cat`?
Why?
What shape does the result have?
What does one row mean?


In [13]:
# Question 3: Answer and reasoning

t = 4
x1 = torch.randn(t)
x2 = torch.randn(t)
x3 = torch.randn(t)
stacked = torch.stack((x1, x2, x3), dim=0)
stacked.shape

# Torch cat concatenats tensors along a decided uppon axis.
# Torch stack on the other hand creates a new dimention, by default at position 0. (first). 
# As the code suggests, stack makes the most sense for the task at hand, as you are looking to create a new dimention, not join at an existing one. 

torch.Size([3, 4])

## Question 4: Debug the bug

Consider:

```python
x = torch.randn(2, 3, 4)
y = x.view(-1, 4)
z = x.transpose(1, 2)
w = z.view(-1, 3)
```

Why can `view` become risky after `transpose`?
What safer alternative is often used instead?
Do not give a full memory-layout derivation.


In [6]:
# Question 4: Answer and reasoning

x = torch.randn(2, 3, 4)
y = x.view(-1, 4)
z = x.transpose(1, 2)
w = z.view(-1, 3) # Error

# View works not by changing the underlying strucutre of the tensor, but changes the 'rules' of how pytorch reads it, with strides.
# When a tensor is transposed, again the underlying strucutre of the data is not changed, just the rules (in which order) to read it.
# So if you transpose then view, you can end up with conflicting rules of how to read the underlyung data. 
# Instead use transpose.reshape.

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

## Question 5: Explain in words

In PyTorch, what is the difference between:
- `expand`
- `repeat`

Keep it simple. One or two lines for each.
If you only know the intuition, give the intuition.


In [ ]:
# Question 5: Answer and reasoning

# Expanding firsly only works if the dimention to be expanded is of size one. It is a fast method that works by setting the underlying stride variable
# to one, effectivly tricking pytorch into reading the same value multiple times. No new memory is allocated for this.

# Repeat on the otherhand does assign new memory. Completing almost the same task as expand but much heavier. But, it can be used on dimentions that 
# are not value=1